# 第七章 PyTorch 小技巧汇总（Practical Tips）

> 适合 Google Colab：边运行、边理解、后续快速复习。  
> 原教程顺序：**7.1 模型保存与加载 → 7.2 Finetune → 7.3 GPU 使用 → 7.4 训练代码模板 → 7.5 TorchMetrics → 7.6 Albumentations → 7.7 TorchEnsemble**

## 学习目标

学完本章，你应能：

1. 正确保存 / 加载 `state_dict`，并理解 checkpoint 与 resume；
2. 区分“冻结 backbone”与“分层学习率”两种 Finetune 策略；
3. 正确管理 CPU / GPU 设备，并理解多 GPU 的现代推荐方案；
4. 写出可复用的训练 / 验证 / 日志 / checkpoint 框架；
5. 理解 TorchMetrics 的“跨 batch 状态累积”；
6. 理解 Albumentations 为什么适合图像 + mask / bbox 的同步增强；
7. 理解模型集成的核心：**多个基模型 + 输出融合**。

## 当前版本更新

本教程主要基于较早 PyTorch 版本，本 Notebook 按当前官方接口修正：

- 推荐保存 **`state_dict`**，而不是 pickle 整个 `nn.Module`。
- PyTorch 2.6 起，`torch.load()` 在未显式传 `pickle_module` 时默认使用 `weights_only=True`；这里显式写出，便于理解与安全加载。
- PyTorch 目前不只支持 CPU / CUDA，还存在 MPS、XPU 等 accelerator；Colab 主线仍以 CUDA 为例。
- 多 GPU：教程重点讲 `nn.DataParallel`；当前官方推荐优先使用 **DistributedDataParallel (DDP)**。
- AMP 新接口使用 `torch.amp.autocast(...)` / `torch.amp.GradScaler(...)`，旧的 `torch.cuda.amp.*` 已弃用。
- TorchMetrics 当前分类 API 通常显式指定 `task="multiclass"`、`num_classes=...`。
- Albumentations、TorchEnsemble 属于第三方库。本 Notebook 的核心代码即使未安装它们也能从上到下运行。

教程：
- https://tingsongyu.github.io/PyTorch-Tutorial-2nd/chapter-7/

官方参考：
- https://docs.pytorch.org/docs/stable/notes/serialization.html
- https://docs.pytorch.org/docs/stable/notes/cuda.html
- https://docs.pytorch.org/docs/stable/amp.html
- https://lightning.ai/docs/torchmetrics/stable/
- https://albumentations.ai/docs/


In [ ]:
import copy
import math
import tempfile
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset

torch.manual_seed(42)
np.random.seed(42)

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

# 7.1 模型保存与加载

序列化（serialization）就是把内存中的训练状态保存到文件；反序列化（deserialization）则把它恢复回来。

PyTorch 最常见的两种思路：

1. 保存整个 `nn.Module`；
2. 保存模型的 `state_dict()`。

**推荐第 2 种。**  
原因是它只保存参数 / buffer，与 Python 类定义解耦程度更高，也更符合 PyTorch 官方最佳实践。

最常见流程：

```text
model.state_dict()
      ↓
torch.save(...)
      ↓
torch.load(..., map_location=...)
      ↓
new_model.load_state_dict(...)
```

In [ ]:
class TinyMLP(nn.Module):
    def __init__(self, in_features=4, hidden=8, num_classes=3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_features, hidden),
            nn.ReLU(),
            nn.Linear(hidden, num_classes),
        )

    def forward(self, x):
        return self.net(x)

model = TinyMLP()
x_demo = torch.randn(2, 4)

with torch.no_grad():
    y_before = model(x_demo)

print("state_dict keys:")
for k in model.state_dict().keys():
    print(" ", k)

## 7.1.1 保存 / 加载 `state_dict`

`map_location="cpu"` 非常实用：即使 checkpoint 原来来自 GPU，也可以先安全加载到 CPU。

当前 PyTorch 中显式写 `weights_only=True` 更清楚，也减少加载不可信 pickle 对象时的安全风险。

In [ ]:
with tempfile.TemporaryDirectory() as tmpdir:
    path = Path(tmpdir) / "tiny_mlp_state.pth"

    torch.save(model.state_dict(), path)

    restored_model = TinyMLP()
    state = torch.load(path, map_location="cpu", weights_only=True)
    restored_model.load_state_dict(state)

    with torch.no_grad():
        y_after = restored_model(x_demo)

    print("same output:", torch.allclose(y_before, y_after))
    print("saved bytes:", path.stat().st_size)

## 7.1.2 Checkpoint 与 Resume

“恢复训练”不能只恢复模型参数。为了真正接着训练，通常还应保存：

- `model.state_dict()`
- `optimizer.state_dict()`
- `scheduler.state_dict()`
- 当前 `epoch / global_step`
- 必要时还包括 scaler、随机数状态、配置等

因为 AdamW 的动量统计、scheduler 当前进度都属于训练状态。

In [ ]:
torch.manual_seed(0)

model_ckpt = TinyMLP()
optimizer_ckpt = torch.optim.AdamW(model_ckpt.parameters(), lr=1e-2)
scheduler_ckpt = torch.optim.lr_scheduler.StepLR(
    optimizer_ckpt, step_size=1, gamma=0.5
)

# 做一个最小训练 step，让 optimizer 真正产生内部 state
x = torch.randn(8, 4)
y = torch.randint(0, 3, (8,))

optimizer_ckpt.zero_grad()
loss = F.cross_entropy(model_ckpt(x), y)
loss.backward()
optimizer_ckpt.step()
scheduler_ckpt.step()

checkpoint = {
    "model": model_ckpt.state_dict(),
    "optimizer": optimizer_ckpt.state_dict(),
    "scheduler": scheduler_ckpt.state_dict(),
    "epoch": 0,
}

with tempfile.TemporaryDirectory() as tmpdir:
    path = Path(tmpdir) / "checkpoint.pth"
    torch.save(checkpoint, path)

    loaded = torch.load(path, map_location="cpu", weights_only=True)

    model_resume = TinyMLP()
    optimizer_resume = torch.optim.AdamW(model_resume.parameters(), lr=1e-2)
    scheduler_resume = torch.optim.lr_scheduler.StepLR(
        optimizer_resume, step_size=1, gamma=0.5
    )

    model_resume.load_state_dict(loaded["model"])
    optimizer_resume.load_state_dict(loaded["optimizer"])
    scheduler_resume.load_state_dict(loaded["scheduler"])
    start_epoch = loaded["epoch"] + 1

    print("resume from epoch:", start_epoch)
    print("restored lr:", optimizer_resume.param_groups[0]["lr"])
    print("optimizer state entries:", len(optimizer_resume.state))

## 7.1.3 `strict=True / False`

`load_state_dict(..., strict=True)` 要求 checkpoint 与模型 key 完全匹配。

微调时常出现“backbone 一样、分类头不同”的情况，此时可只加载匹配部分；但 `strict=False` **不是自动帮你解决 shape 不匹配**，仍应明确知道哪些 key 缺失 / 多余。

In [ ]:
source = TinyMLP(num_classes=3)
source_state = source.state_dict()

# 模拟：只迁移前两层（feature extractor）
feature_state = {
    k: v for k, v in source_state.items()
    if k.startswith("net.0.")
}

target = TinyMLP(num_classes=2)
result = target.load_state_dict(feature_state, strict=False)

print("missing keys:", result.missing_keys)
print("unexpected keys:", result.unexpected_keys)

# 7.2 Finetune 模型微调

Finetune（微调）属于迁移学习（Transfer Learning）：把源任务中学到的参数作为目标任务的起点。

原教程将网络分为：

```text
feature extractor / backbone → classifier / head
```

两种常用策略：

1. **冻结 backbone，只训练 head**；
2. **backbone 小学习率，head 大学习率**。

这两种思想在 LLM 中仍然存在：全参数微调、部分冻结、参数高效微调（PEFT / LoRA）本质上都在回答“哪些参数参与更新、用多大的更新强度”。

In [ ]:
class TransferNet(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.backbone = nn.Sequential(
            nn.Linear(8, 16),
            nn.ReLU(),
            nn.Linear(16, 16),
            nn.ReLU(),
        )
        self.head = nn.Linear(16, num_classes)

    def forward(self, x):
        return self.head(self.backbone(x))

# 模拟一个已经训练好的源模型
source_model = TransferNet(num_classes=4)

# 目标任务类别数不同：只迁移 backbone
target_model = TransferNet(num_classes=2)
target_model.backbone.load_state_dict(source_model.backbone.state_dict())

print("source head:", source_model.head)
print("target head:", target_model.head)

## 7.2.1 方法一：冻结 Backbone

核心机制就是：

```python
param.requires_grad = False
```

推荐优化器只接收 `requires_grad=True` 的参数，这样意图更清晰。

In [ ]:
for p in target_model.backbone.parameters():
    p.requires_grad = False

optimizer_head_only = torch.optim.AdamW(
    (p for p in target_model.parameters() if p.requires_grad),
    lr=1e-3,
)

trainable = [
    name for name, p in target_model.named_parameters()
    if p.requires_grad
]

print("trainable parameters:", trainable)
print("optimizer parameter tensors:",
      sum(len(g["params"]) for g in optimizer_head_only.param_groups))

## 7.2.2 方法二：分层学习率（Discriminative Learning Rates）

如果 backbone 也允许继续适应目标任务，可以给它更小的学习率，而新 head 使用更大学习率。

这依赖第五章的 `optimizer.param_groups`。

In [ ]:
target_model_2 = TransferNet(num_classes=2)
target_model_2.backbone.load_state_dict(source_model.backbone.state_dict())

optimizer_groups = torch.optim.AdamW([
    {"params": target_model_2.backbone.parameters(), "lr": 1e-4},
    {"params": target_model_2.head.parameters(), "lr": 1e-3},
], weight_decay=1e-2)

for i, group in enumerate(optimizer_groups.param_groups):
    print(f"group {i}: lr={group['lr']}, tensors={len(group['params'])}")

### Finetune 时容易犯的错误

- 换了 head，却把旧 head 的参数也强行加载；
- `requires_grad=False` 后又忘了检查哪些参数真正可训练；
- 冻结参数但仍让 BatchNorm 等模块保持训练行为——是否需要 `eval()` 取决于具体策略；
- 学习率对所有层一视同仁，导致 pretrained 表征被快速破坏；
- 把“加载预训练权重”误认为“已经完成微调”——微调仍需要目标任务训练。

# 7.3 GPU 使用

PyTorch 运算最重要的设备规则：

> **参与同一次算子的 Tensor 必须位于兼容的设备上；模型参数和输入通常应迁移到同一 device。**

Colab 最常见：

```python
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)
x = x.to(device)
```

注意：Tensor 的 `.to()` 返回 Tensor，因此通常要重新赋值；`nn.Module.to()` 会迁移模块的参数和 buffer，并返回模块自身。

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

device_model = nn.Linear(4, 2).to(device)
device_x = torch.randn(3, 4).to(device)
device_y = device_model(device_x)

print("device:", device)
print("model parameter device:", next(device_model.parameters()).device)
print("input device:", device_x.device)
print("output device:", device_y.device)

## 7.3.1 CUDA 常用检查

教程列出了 `device_count()`、`get_device_name()`、`mem_get_info()`、`memory_summary()`、`empty_cache()` 等。

需要特别注意：

- `empty_cache()` 释放的是 **PyTorch caching allocator 中未占用的缓存块**，不是“把当前正在使用的 Tensor 显存清空”；
- `cudnn.benchmark=True` 更适合输入 shape 稳定、追求速度的卷积任务；
- 强可复现要求与极致性能往往存在取舍。

In [ ]:
if torch.cuda.is_available():
    print("GPU count:", torch.cuda.device_count())
    print("GPU 0:", torch.cuda.get_device_name(0))
    free_bytes, total_bytes = torch.cuda.mem_get_info(0)
    print(f"free / total: {free_bytes/2**30:.2f} / {total_bytes/2**30:.2f} GiB")
else:
    print("当前环境没有 CUDA；Colab 切换到 GPU runtime 后本单元会显示 GPU 信息。")

## 7.3.2 混合精度 AMP：当前推荐 API

对 Transformer / LLM 很重要。AMP（Automatic Mixed Precision）核心是：

- forward 中对适合的算子使用较低精度；
- fp16 训练常配合 gradient scaling，降低梯度 underflow 风险；
- 当前推荐 `torch.amp.autocast` / `torch.amp.GradScaler`，而不是旧的 `torch.cuda.amp.*`。

下面的代码在 CPU 环境也能运行；有 CUDA 时会自动使用 CUDA AMP。

In [ ]:
amp_model = nn.Linear(8, 2).to(device)
amp_optimizer = torch.optim.AdamW(amp_model.parameters(), lr=1e-3)
amp_x = torch.randn(16, 8, device=device)
amp_target = torch.randint(0, 2, (16,), device=device)

# CUDA 用 float16；CPU 用 bfloat16，仅用于演示 autocast 机制
amp_dtype = torch.float16 if device.type == "cuda" else torch.bfloat16
scaler = torch.amp.GradScaler("cuda", enabled=(device.type == "cuda"))

amp_optimizer.zero_grad()

with torch.amp.autocast(device_type=device.type, dtype=amp_dtype):
    amp_logits = amp_model(amp_x)
    amp_loss = F.cross_entropy(amp_logits, amp_target)

scaler.scale(amp_loss).backward()
scaler.step(amp_optimizer)
scaler.update()

print("loss:", float(amp_loss))
print("autocast output dtype:", amp_logits.dtype)
print("GradScaler enabled:", scaler.is_enabled())

## 7.3.3 多 GPU：理解 DataParallel，但新项目优先 DDP

教程用 `nn.DataParallel` 解释了 4 个阶段：

```text
scatter 输入
→ replicate 模型
→ parallel_apply
→ gather 输出
```

这个机制仍值得理解，但当前 PyTorch 官方明确建议多数多 GPU 训练使用 **DistributedDataParallel (DDP)**，即通常“一张 GPU 一个进程”。

| 方案 | 核心特点 | 当前建议 |
|---|---|---|
| `DataParallel` | 单进程、多线程、每轮复制模型 | 了解旧项目即可 |
| `DistributedDataParallel` | 多进程，每 GPU 一个进程 | 新项目优先 |
| FSDP 等 | 参数 / 梯度 / optimizer state 分片 | 大模型进一步学习 |

### `CUDA_VISIBLE_DEVICES`

服务器上常通过环境变量限制进程可见 GPU。它应该在初始化 CUDA **之前**设置。

例如 shell 中：

```bash
CUDA_VISIBLE_DEVICES=2,3 python train.py
```

此时物理 GPU 2、3 在程序内部通常重新编号为逻辑 `cuda:0`、`cuda:1`。

In [ ]:
print("visible CUDA device count:", torch.cuda.device_count())

# 仅展示 DataParallel 的安全写法；单卡/CPU 环境不会包装
parallel_demo = nn.Linear(4, 2).to(device)
if torch.cuda.device_count() > 1:
    parallel_demo = nn.DataParallel(parallel_demo)
    print("wrapped by DataParallel")
else:
    print("<= 1 GPU：跳过 DataParallel；多卡新项目应优先学习 DDP。")

## 7.3.4 `module.` 前缀问题

旧式 `DataParallel` 保存的 `state_dict` 往往带有 `module.` 前缀。直接加载到未包装模型会出现 missing / unexpected keys。

遇到旧 checkpoint 时，可在**确认 checkpoint 来源可信、结构明确**后统一去掉前缀。

In [ ]:
fake_dp_state = {
    "module.weight": torch.randn(2, 4),
    "module.bias": torch.randn(2),
}

clean_state = {
    (k.removeprefix("module.")): v
    for k, v in fake_dp_state.items()
}

plain_model = nn.Linear(4, 2)
plain_model.load_state_dict(clean_state)

print("clean keys:", list(clean_state.keys()))

# 7.4 模型训练代码模板

教程提炼了四个工程模块：

1. 参数 / 超参数配置；
2. logging / TensorBoard 日志；
3. `train_one_epoch()` 与 `evaluate()`；
4. checkpoint 保存。

在 `.py` 训练脚本中，`argparse` 很适合命令行实验；Notebook 中用 `dataclass` 更直观。两者解决的是同一个问题：

> **不要把关键超参数散落在代码各处。**

下面构造一套最小但完整的训练骨架。

In [ ]:
@dataclass
class TrainConfig:
    input_dim: int = 8
    num_classes: int = 3
    batch_size: int = 32
    epochs: int = 4
    lr: float = 1e-2
    seed: int = 42

cfg = TrainConfig()
print(cfg)

## 7.4.1 `AverageMeter`

训练中不能简单“把每个 batch 的平均 loss 再做平均”——最后一个 batch 大小时可能不同。

正确做法是按样本数加权：

$$
\bar{x} = \frac{\sum_i x_i n_i}{\sum_i n_i}
$$

In [ ]:
class AverageMeter:
    def __init__(self):
        self.reset()

    def reset(self):
        self.sum = 0.0
        self.count = 0

    def update(self, value, n=1):
        self.sum += float(value) * n
        self.count += n

    @property
    def avg(self):
        return self.sum / max(self.count, 1)

meter = AverageMeter()
meter.update(2.0, n=3)
meter.update(1.0, n=1)
print("weighted average:", meter.avg)  # (2*3 + 1*1) / 4 = 1.75

## 7.4.2 `train_one_epoch` / `evaluate`

要点：

- 训练：`model.train()` + 梯度更新；
- 验证：`model.eval()` + `torch.inference_mode()`；
- 验证阶段不应调用 `optimizer.step()`；
- loss 要按样本量正确累计。

In [ ]:
def make_synthetic_classification(n=320, input_dim=8, num_classes=3, seed=42):
    g = torch.Generator().manual_seed(seed)
    x = torch.randn(n, input_dim, generator=g)
    teacher = torch.randn(input_dim, num_classes, generator=g)
    y = (x @ teacher).argmax(dim=1)
    return x, y

def train_one_epoch(model, loader, optimizer, device):
    model.train()
    loss_meter = AverageMeter()
    correct = 0
    total = 0

    for x, y in loader:
        x, y = x.to(device), y.to(device)

        optimizer.zero_grad()
        logits = model(x)
        loss = F.cross_entropy(logits, y)
        loss.backward()
        optimizer.step()

        batch_size = y.size(0)
        loss_meter.update(loss.item(), batch_size)
        correct += (logits.argmax(dim=1) == y).sum().item()
        total += batch_size

    return {
        "loss": loss_meter.avg,
        "acc": correct / total,
    }

@torch.inference_mode()
def evaluate(model, loader, device):
    model.eval()
    loss_meter = AverageMeter()
    correct = 0
    total = 0

    for x, y in loader:
        x, y = x.to(device), y.to(device)
        logits = model(x)
        loss = F.cross_entropy(logits, y)

        batch_size = y.size(0)
        loss_meter.update(loss.item(), batch_size)
        correct += (logits.argmax(dim=1) == y).sum().item()
        total += batch_size

    return {
        "loss": loss_meter.avg,
        "acc": correct / total,
    }

## 7.4.3 完整最小训练 + Best Checkpoint

真实项目还可以加入：

- Python `logging`
- TensorBoard / W&B
- AMP
- gradient clipping
- early stopping
- distributed training
- 配置文件与实验追踪

但核心控制流不应被这些功能淹没。

In [ ]:
torch.manual_seed(cfg.seed)

x_all, y_all = make_synthetic_classification(
    n=320,
    input_dim=cfg.input_dim,
    num_classes=cfg.num_classes,
    seed=cfg.seed,
)

train_ds = TensorDataset(x_all[:256], y_all[:256])
val_ds = TensorDataset(x_all[256:], y_all[256:])

train_loader = DataLoader(train_ds, batch_size=cfg.batch_size, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=cfg.batch_size)

train_model = TinyMLP(
    in_features=cfg.input_dim,
    hidden=32,
    num_classes=cfg.num_classes,
).to(device)

train_optimizer = torch.optim.AdamW(train_model.parameters(), lr=cfg.lr)
best_acc = -1.0
best_state = None
history = []

for epoch in range(cfg.epochs):
    train_metrics = train_one_epoch(
        train_model, train_loader, train_optimizer, device
    )
    val_metrics = evaluate(train_model, val_loader, device)

    history.append((train_metrics, val_metrics))

    if val_metrics["acc"] > best_acc:
        best_acc = val_metrics["acc"]
        # clone 到 CPU，避免后续参数更新影响“最佳权重快照”
        best_state = {
            k: v.detach().cpu().clone()
            for k, v in train_model.state_dict().items()
        }

    print(
        f"epoch {epoch+1}: "
        f"train_loss={train_metrics['loss']:.4f}, "
        f"val_acc={val_metrics['acc']:.3f}"
    )

print("best val acc:", best_acc)
assert best_state is not None

### 7.4 对 LLM 训练最重要的迁移

以后看 Transformer / LLM 训练框架时，仍然是在扩展同一骨架：

```text
config
→ dataset / dataloader
→ model
→ optimizer / scheduler
→ train step
→ eval
→ log
→ checkpoint / resume
```

只是会额外出现：

- gradient accumulation；
- AMP / BF16；
- gradient clipping；
- DDP / FSDP / ZeRO；
- tokenizer / sequence packing；
- checkpoint sharding；
- 更复杂的 metric 与 generation evaluation。

# 7.5 TorchMetrics 模型评估指标库

TorchMetrics 的核心价值不是“帮你写一个 accuracy 公式”，而是：

> **自动维护跨 batch 的 metric state，并可在分布式环境同步。**

典型生命周期：

```text
metric.update(...)  # 多个 batch
      ↓
metric.compute()    # 整个 epoch
      ↓
metric.reset()      # 下一轮重新开始
```

当前分类接口通常要明确任务类型，例如：

```python
Accuracy(task="multiclass", num_classes=3)
```

Notebook 不强制安装第三方库；未安装时会给出提示，其余单元仍可运行。

In [ ]:
try:
    import torchmetrics
    from torchmetrics.classification import MulticlassAccuracy
    HAVE_TORCHMETRICS = True
    print("torchmetrics:", torchmetrics.__version__)
except ImportError:
    HAVE_TORCHMETRICS = False
    print("未安装 torchmetrics。可选安装：%pip install -q torchmetrics")

In [ ]:
tm_preds_1 = torch.tensor([
    [3.0, 1.0, 0.0],
    [0.2, 2.0, 0.1],
])
tm_target_1 = torch.tensor([0, 1])

tm_preds_2 = torch.tensor([
    [0.1, 0.2, 2.5],
    [2.0, 1.0, 0.0],
])
tm_target_2 = torch.tensor([2, 2])

if HAVE_TORCHMETRICS:
    metric = MulticlassAccuracy(num_classes=3, average="micro")

    metric.update(tm_preds_1, tm_target_1)
    metric.update(tm_preds_2, tm_target_2)

    print("epoch accuracy:", float(metric.compute()))
    metric.reset()
    print("state reset 完成")
else:
    preds = torch.cat([tm_preds_1, tm_preds_2]).argmax(dim=1)
    target = torch.cat([tm_target_1, tm_target_2])
    print("manual epoch accuracy:", float((preds == target).float().mean()))

## 7.5.1 自定义 Metric 的抽象

教程强调自定义 TorchMetrics 时的三个核心点：

- `add_state(...)`：注册需要跨 batch 累积的状态；
- `update(...)`：接收一个 batch，更新状态；
- `compute()`：从累计状态计算最终指标。

这和 `nn.Module` 的设计思想相似：框架管理通用生命周期，你实现任务特有逻辑。

对于 LLM，不要把所有评估都压缩成 token accuracy。实际还可能关注：

- perplexity；
- exact match；
- ROUGE / BLEU（特定任务）；
- 生成质量、事实性、安全性等任务级评估。

# 7.6 Albumentations 数据增强库

教程强调 Albumentations 的优势：**图像与标签可以同步做几何变换**。

对于分类，图像变了但类别通常不变；但对于：

- 语义分割：image 与 mask 必须同步；
- 目标检测：image 与 bbox 必须同步；
- 姿态估计：image 与 keypoints 必须同步。

Albumentations 的基本契约：

```text
named targets 输入
→ Compose / transform
→ dict 输出
```

例如：

```python
result = transform(image=image, mask=mask)
image2 = result["image"]
mask2 = result["mask"]
```

这节与 LLM 关联较弱，理解“多模态目标同步变换”的工程思想即可。

In [ ]:
try:
    import albumentations as A
    HAVE_ALBUMENTATIONS = True
    print("albumentations:", A.__version__)
except ImportError:
    HAVE_ALBUMENTATIONS = False
    print("未安装 albumentations。可选安装：%pip install -q albumentations")

## 7.6.1 为什么 image 和 mask 必须同步？

下面不用任何第三方库，直接构造一个 4×4 图像和分割 mask。

若只翻转 image、不翻转 mask，监督信号就错位。

In [ ]:
image = np.array([
    [0, 0, 0, 9],
    [0, 0, 0, 9],
    [0, 0, 0, 9],
    [0, 0, 0, 9],
], dtype=np.uint8)

mask = np.array([
    [0, 0, 0, 1],
    [0, 0, 0, 1],
    [0, 0, 0, 1],
    [0, 0, 0, 1],
], dtype=np.uint8)

flipped_image = np.fliplr(image)
flipped_mask = np.fliplr(mask)

print("original image:\n", image)
print("original mask:\n", mask)
print("flipped image:\n", flipped_image)
print("flipped mask:\n", flipped_mask)

# 亮区域和前景 mask 仍然对齐
print("still aligned:", np.array_equal(flipped_image > 0, flipped_mask > 0))

## 7.6.2 Albumentations 的 `Compose`

如果环境已安装 Albumentations，下方直接执行真实 API；未安装则安全跳过，不影响整本 Notebook。

当前 API 中仍应使用**命名参数**传入 target，输出仍是字典。

In [ ]:
if HAVE_ALBUMENTATIONS:
    transform = A.Compose([
        A.HorizontalFlip(p=1.0),
    ])

    image_rgb = np.repeat(image[..., None], 3, axis=2)
    result = transform(image=image_rgb, mask=mask)

    aug_image = result["image"]
    aug_mask = result["mask"]

    print("image shape:", aug_image.shape)
    print("mask shape:", aug_mask.shape)
    print(
        "aligned:",
        np.array_equal(aug_image[..., 0] > 0, aug_mask > 0)
    )
else:
    print("跳过真实 Albumentations API；上一个单元已演示同步空间变换的核心机制。")

### Albumentations 代码结构：只记抽象

原教程进一步分析了其类层次。无需记源码行号，只需理解：

```text
BasicTransform
├─ ImageOnlyTransform   # 只改图像内容，例如亮度、模糊
└─ DualTransform        # 空间变换，需要同步 image / mask / bbox / keypoint
```

真正重要的不是某个类名，而是：

> **空间变换会改变坐标，所以与图像共享坐标系的标签必须使用同一组随机参数。**

# 7.7 TorchEnsemble 模型集成库

Ensemble（模型集成）的核心不是某个库，而是：

$$
f(x)=\operatorname{Combine}\left(f_1(x),f_2(x),\dots,f_M(x)\right)
$$

教程介绍 TorchEnsemble 的 Fusion、Voting、Bagging、Gradient Boosting、Snapshot Ensemble 等方法。

最值得掌握的差异：

- **Fusion**：先融合模型输出（如 logits），再转概率；
- **Soft Voting**：每个模型先转概率，再平均；
- **Bagging**：除了输出融合，还通过不同采样让基模型产生差异；
- **Boosting**：后续学习器重点修正前面学习器的误差；
- **Snapshot Ensemble**：一次训练过程中保存多个不同阶段的模型快照做集成。

TorchEnsemble 是第三方库，且发布节奏与 PyTorch 本体不同。工程上应优先掌握手动集成机制，而不是依赖某个封装库。

In [ ]:
try:
    import torchensemble
    HAVE_TORCHENSEMBLE = True
    print("torchensemble imported")
except ImportError:
    HAVE_TORCHENSEMBLE = False
    print("未安装 torchensemble。可选安装：%pip install -q torchensemble")

## 7.7.1 用 `nn.ModuleList` 手动实现 Soft Voting

`nn.ModuleList` 很关键：与普通 Python `list` 不同，它会把子模型注册为 Module，从而使参数能被 `state_dict()`、`.to(device)` 等框架机制正确管理。

下面对多个分类模型的 softmax 概率取平均。

In [ ]:
class SoftVotingEnsemble(nn.Module):
    def __init__(self, models):
        super().__init__()
        self.models = nn.ModuleList(models)

    def forward(self, x):
        probs = [
            F.softmax(model(x), dim=-1)
            for model in self.models
        ]
        return torch.stack(probs, dim=0).mean(dim=0)

ensemble = SoftVotingEnsemble([
    TinyMLP(in_features=4, hidden=8, num_classes=3),
    TinyMLP(in_features=4, hidden=8, num_classes=3),
    TinyMLP(in_features=4, hidden=8, num_classes=3),
])

ensemble_x = torch.randn(5, 4)
ensemble_prob = ensemble(ensemble_x)

print("output shape:", ensemble_prob.shape)
print("row sums:", ensemble_prob.sum(dim=1))
print("registered parameter tensors:", len(list(ensemble.parameters())))

## 7.7.2 “平均 logits”与“平均概率”不是同一件事

Fusion 常见做法：

$$
p = \operatorname{softmax}\left(\frac{1}{M}\sum_m z_m\right)
$$

Soft Voting：

$$
p = \frac{1}{M}\sum_m \operatorname{softmax}(z_m)
$$

由于 softmax 是非线性的，两者一般不相等。

In [ ]:
logits_a = torch.tensor([[4.0, 1.0, 0.0]])
logits_b = torch.tensor([[0.0, 2.0, 2.0]])

fusion_prob = F.softmax((logits_a + logits_b) / 2, dim=-1)
voting_prob = (
    F.softmax(logits_a, dim=-1)
    + F.softmax(logits_b, dim=-1)
) / 2

print("softmax(mean logits):", fusion_prob)
print("mean probabilities  :", voting_prob)
print("same:", torch.allclose(fusion_prob, voting_prob))

### 7.7 对 LLM 的连接

今天的大模型也大量使用“ensemble 思想”，但形式不一定是训练多个完整模型，例如：

- 多模型投票 / reranking；
- 多次采样后 self-consistency；
- 多个 checkpoint / adapter 的结果融合；
- Mixture-of-Experts（MoE）虽然不是传统 ensemble，但同样体现“多个专家 + 路由 / 聚合”的思想。

概念上要分清：**ensemble 的核心是多个预测器的组合；不是简单把参数文件拼起来。**

# 本章知识结构总结

```text
7.1 Serialization
├─ state_dict
├─ torch.save / torch.load
├─ map_location
└─ checkpoint / resume

7.2 Finetune
├─ load pretrained backbone
├─ freeze backbone
└─ discriminative learning rates

7.3 GPU
├─ model / tensor → same device
├─ CUDA diagnostics
├─ AMP
├─ DataParallel（理解旧机制）
└─ DDP（当前多卡主线）

7.4 Training Template
├─ config
├─ train_one_epoch
├─ evaluate
├─ AverageMeter / metrics
└─ best checkpoint

7.5 TorchMetrics
├─ update
├─ compute
└─ reset

7.6 Albumentations
├─ Compose
├─ named targets
└─ image / mask / bbox / keypoints 同步变换

7.7 Ensemble
├─ Fusion
├─ Voting
├─ Bagging / Boosting
└─ ModuleList 手动集成
```

## 对 Transformer / LLM 的优先级

**必须掌握：**

1. `state_dict`、checkpoint、resume；
2. Finetune 的参数冻结与参数组；
3. device、AMP、DDP 的基本概念；
4. 可复用的 train/eval/checkpoint 训练框架。

**了解即可：**

5. TorchMetrics 的状态化指标思想；
6. Albumentations（偏 CV）；
7. TorchEnsemble（理解 ensemble 思想比记库 API 更重要）。

# 学完必须会回答的问题

1. 为什么推荐保存 `state_dict`，而不是直接保存整个 `nn.Module`？
2. `map_location="cpu"` 解决什么问题？
3. Resume 训练时为什么不能只恢复模型权重？
4. PyTorch 2.6+ 的 `weights_only=True` 与加载安全有什么关系？
5. Finetune 中“冻结 backbone”和“backbone 小学习率”分别适合什么思路？
6. `requires_grad=False` 与 `model.eval()` 是同一件事吗？为什么？
7. 为什么模型参数和输入 Tensor 通常必须位于同一个 device？
8. `DataParallel` 的 scatter → replicate → parallel_apply → gather 各做什么？
9. 为什么当前多 GPU 新项目通常优先 DDP 而不是 DataParallel？
10. AMP 中 autocast 与 GradScaler 分别解决什么问题？
11. `train_one_epoch()` 与 `evaluate()` 在模式、梯度和参数更新上有什么区别？
12. 为什么 AverageMeter 应按样本数加权，而不是简单平均各 batch loss？
13. TorchMetrics 的 `update → compute → reset` 为什么适合 epoch 级指标？
14. 为什么分割任务的数据增强必须同步变换 image 与 mask？
15. 为什么 `softmax(mean(logits))` 通常不等于 `mean(softmax(logits))`？
